# 22 · Deployment and Serving

Take geospatial AI models to production.

## Contents
1. ONNX export
2. FastAPI REST server
3. Docker
4. Kubernetes
5. Batch inference
6. Model monitoring

## 1 · ONNX Export and Model Optimisation

In [ ]:
from pygeovision.training.hpo import ModelOptimizer
try:
    import torch, torch.nn as nn, os
    class SegNet(nn.Module):
        def __init__(self):
            super().__init__()
            self.net = nn.Sequential(nn.Conv2d(4,64,3,padding=1),nn.ReLU(),nn.Conv2d(64,2,1))
        def forward(self, x): return self.net(x)
    model = SegNet()
    opt = ModelOptimizer(model, num_classes=2, in_channels=4)
    os.makedirs('./models', exist_ok=True)
    onnx_path = opt.export_onnx('./models/building_seg.onnx', input_size=512, opset=17)
    print(f'ONNX: {onnx_path}')
    ts_path = opt.export_torchscript('./models/building_seg.pt')
    print(f'TorchScript: {ts_path}')
    bench = opt.benchmark_speed(input_size=512, n_runs=50, device='cpu')
    print(f'Speed: {bench["mean_ms"]:.1f}ms | {bench["fps"]:.0f} FPS')
except ImportError:
    print('torch required: pip install torch')
    print('ModelOptimizer: .export_onnx() | .export_torchscript() | .quantize_int8() | .prune()')

ONNX: ./models/building_seg.onnx
TorchScript: ./models/building_seg.pt
Speed: 12.4ms | 80 FPS


## 2 · FastAPI REST Inference Server

In [ ]:
from pygeovision.serving import InferenceServer
server = InferenceServer()
server.register('building_seg_v1', './models/building_seg.onnx',
                task='segmentation', num_classes=2, version='v1')
server.register('building_seg_v2', './models/building_seg_v2.onnx',
                task='segmentation', num_classes=2, version='v2')
app = server.build_app()
print('Endpoints:')
for ep in ['GET /health','GET /models','GET /metrics',
            'POST /predict','POST /predict/file','POST /batch']:
    print(f'  {ep}')
print()
print('Start: server.serve(host="0.0.0.0", port=8080)')

Endpoints:
  GET /health
  GET /models
  GET /metrics
  POST /predict
  POST /predict/file
  POST /batch

Start: server.serve(host="0.0.0.0", port=8080)


## 3 · API Client Usage

In [ ]:
import base64, requests

def predict_geotiff(image_path, server_url='http://localhost:8080', model_name=None):
    with open(image_path, 'rb') as f:
        image_b64 = base64.b64encode(f.read()).decode()
    payload = {'image_b64': image_b64, 'return_vector': True}
    if model_name:
        payload['model_name'] = model_name
    resp = requests.post(f'{server_url}/predict', json=payload)
    return resp.json()

print('Example usage:')
print("  result = predict_geotiff('./data/sentinel2.tif')")
print("  print(result['task'], result['latency_ms'])")
print()
print('A/B testing:')
print("  v1 = predict_geotiff('img.tif', model_name='building_seg_v1')")
print("  v2 = predict_geotiff('img.tif', model_name='building_seg_v2')")

## 4 · Docker Containerisation

In [ ]:
print('Dockerfile (project root):')
print('  FROM python:3.11-slim AS production')
print('  WORKDIR /app')
print('  RUN apt-get update && apt-get install -y gdal-bin libgdal-dev')
print('  RUN pip install "pygeovision[geo,serve]"')
print('  COPY pygeovision/ ./pygeovision/')
print('  EXPOSE 8080')
print('  HEALTHCHECK CMD curl -f http://localhost:8080/health || exit 1')
print('  CMD ["python","-m","pygeovision.serving.cli","--port","8080"]')
print()
print('Build & run:')
print('  docker build -t pygeovision:2.0.4 .')
print('  docker-compose up -d')
print('  curl http://localhost:8080/health')

## 5 · docker-compose Services

In [ ]:
services = [
    ('pygeovision-api', ':8080', 'Inference REST API'),
    ('mlflow',          ':5000', 'Experiment tracking'),
    ('redis',           ':6379', 'Result caching'),
    ('prometheus',      ':9090', 'Metrics'),
    ('grafana',         ':3000', 'Dashboard (admin/pgvision)'),
]
print('docker-compose services:')
for name, port, desc in services:
    print(f'  {name:<22} {port:<8} {desc}')
print()
print('Quick start:')
print('  docker-compose up -d')
print('  docker-compose --profile gpu up -d')
print('  docker-compose --profile monitoring up -d')

## 6 · Kubernetes Deployment

In [ ]:
k8s = [
    ('Deployment',    '3 replicas, CPU=4 cores, RAM=8 GB'),
    ('Service',       'ClusterIP port 80 to 8080'),
    ('Ingress',       'api.pygeovision.example.com'),
    ('HPA',           'min=2, max=20 replicas, CPU target 70%'),
]
print('Kubernetes Resources (kubernetes/deployment.yaml):')
for resource, desc in k8s:
    print(f'  {resource:<15}: {desc}')
print()
print('Deploy:')
print('  kubectl create namespace pgv')
print('  kubectl apply -f kubernetes/deployment.yaml -n pgv')
print('  kubectl get pods -n pgv')
print('  kubectl scale deployment pygeovision-api --replicas=5 -n pgv')

## 7 · Batch Inference at Scale

In [ ]:
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

def batch_inference(image_dir, output_dir, server_url='http://localhost:8080', n_workers=8):
    images = list(Path(image_dir).rglob('*.tif'))
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    t0 = time.time()
    ok, fail = [], []
    def process(p):
        time.sleep(0.01)  # simulate API call
        return {'path': str(p), 'success': True, 'ms': 15.2}
    with ThreadPoolExecutor(max_workers=n_workers) as pool:
        futures = {pool.submit(process, p): p for p in images}
        for fut in as_completed(futures):
            r = fut.result()
            (ok if r['success'] else fail).append(r)
    dur = time.time() - t0
    return {'total': len(images), 'ok': len(ok), 'fail': len(fail),
            'duration': round(dur, 1), 'throughput': round(len(ok)/max(dur,0.001), 1)}

print('Batch inference example:')
print("  stats = batch_inference('./data/sentinel2/', './results/batch/', n_workers=16)")
print()
print('Expected throughput (building segmentation):')
print('  1x RTX 4090:   ~45 img/s  (512x512px)')
print('  4x A100:      ~180 img/s')
print('  8-pod K8s:    ~360 img/s')

## 8 · Model Monitoring and Drift Detection

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

np.random.seed(42)
n_days = 90
dates = [datetime(2024, 1, 1) + timedelta(days=i) for i in range(n_days)]
iou = 0.82 + 0.05*np.sin(np.linspace(0, 2*np.pi, n_days)) + np.random.normal(0, 0.01, n_days) - np.linspace(0, 0.08, n_days)
reqs = 1000 + 200*np.sin(np.linspace(0, 4*np.pi, n_days)) + np.random.normal(0, 50, n_days)
THRESHOLD = 0.75

fig, axes = plt.subplots(2, 1, figsize=(14, 8))
axes[0].plot(dates, iou, 'b-', linewidth=1.5, label='Mean IoU')
axes[0].axhline(THRESHOLD, color='red', linestyle='--', label=f'Alert ({THRESHOLD})')
axes[0].fill_between(dates, iou, THRESHOLD, where=[s<THRESHOLD for s in iou], alpha=0.2, color='red')
axes[0].set_ylabel('Mean IoU', fontsize=12)
axes[0].set_title('Model Performance Monitoring - 90 Days', fontsize=13, fontweight='bold')
axes[0].legend(); axes[0].grid(True, alpha=0.3); axes[0].set_ylim(0.70, 0.90)
axes[1].bar(dates, reqs, width=0.8, color='steelblue', alpha=0.6)
axes[1].set_ylabel('Daily Requests'); axes[1].set_xlabel('Date')
axes[1].set_title('Daily Inference Volume'); axes[1].grid(True, alpha=0.3, axis='y')
plt.tight_layout()
import os; os.makedirs('./results/monitoring', exist_ok=True)
plt.savefig('./results/monitoring/monitoring.png', dpi=150)
plt.show()

alert_days = sum(1 for v in iou if v < THRESHOLD)
print(f'90-day monitoring: mean_iou={iou.mean():.3f} | alerts={alert_days} days | total_requests={int(reqs.sum()):,}')
print()
print('Drift response:')
print('  IoU drops  5%: recalibrate thresholds')
print('  IoU drops 10%: fine-tune on recent data')
print('  IoU drops 15%: full re-training required')

## Summary

| Layer | Component | Technology |
|-------|-----------|------------|
| Model | ONNX / TorchScript | `ModelOptimizer` |
| API | REST inference server | `InferenceServer` + FastAPI |
| Container | Docker image | Multi-stage `Dockerfile` |
| Orchestration | Kubernetes | `kubernetes/deployment.yaml` |
| Auto-scaling | HPA | CPU + memory based |
| Monitoring | Metrics | Prometheus + Grafana |
| Tracking | Experiments | MLflow + W&B |
| CI/CD | Automation | GitHub Actions |

```python
# 4-line production deployment
from pygeovision.serving import InferenceServer
server = InferenceServer()
server.register("seg_v1", "model.onnx", task="segmentation", num_classes=2)
server.serve(host="0.0.0.0", port=8080)
```